# DeepFake Detection — Celeb-DF v2 Full Pipeline
### EfficientNet-Lite0 + LSTM | Trained on Celeb-DF v2

**Pipeline Overview:**
1. Mount Drive & install dependencies
2. Unzip Celeb-DF v2
3. Preprocess videos (MTCNN face extraction)
4. Split into train / val / test
5. Explore & visualize the dataset
6. Train EfficientNet-Lite0 + LSTM model
7. Evaluate with full metrics & visualizations
8. Predict on any new video

> **Before running:** Runtime → Change runtime type → **T4 GPU**
>
> **Required:** `Celeb-DF-v2.zip` must be in `MyDrive/Celeb-DF-v2.zip`

## Step 1 — Setup: Mount Drive & Install Dependencies

In [1]:
from google.colab import drive
drive.mount('/content/drive')

!pip install -q timm
print("All dependencies installed!")

NotImplementedError: Mounting drive is unsupported in this environment. Use PyDrive2 instead. See examples at https://colab.research.google.com/notebooks/io.ipynb#scrollTo=7taylj9wpsA2.

## Step 2 — Unzip Celeb-DF v2
> Skip if already unzipped.

In [ ]:
import os, zipfile

ZIP_PATH    = "/content/drive/MyDrive/Celeb-DF-v2.zip"
EXTRACT_DIR = "/content/drive/MyDrive/celeb_df_raw"

if not os.path.exists(EXTRACT_DIR):
    os.makedirs(EXTRACT_DIR, exist_ok=True)
    print("Unzipping... this may take 20-30 minutes to Drive.")
    with zipfile.ZipFile(ZIP_PATH, 'r') as z:
        members = z.namelist()
        for i, member in enumerate(members):
            z.extract(member, EXTRACT_DIR)
            if (i+1) % 500 == 0:
                print(f"  Extracted {i+1}/{len(members)} files...")
    print("Unzip complete!")
else:
    print("Already unzipped, skipping.")

print("\nContents:")
base = EXTRACT_DIR
contents = os.listdir(base)
if len(contents) == 1 and os.path.isdir(os.path.join(base, contents[0])):
    base = os.path.join(base, contents[0])
for item in sorted(os.listdir(base)):
    full = os.path.join(base, item)
    if os.path.isdir(full):
        n = len(os.listdir(full))
        print(f"  {item}/  ->  {n} items")
    else:
        print(f"  {item}")

## Step 3 — Detect & Merge Dataset Paths

In [ ]:
import os, shutil

base = "/content/drive/MyDrive/celeb_df_raw"
contents = os.listdir(base)
if len(contents) == 1 and os.path.isdir(os.path.join(base, contents[0])):
    base = os.path.join(base, contents[0])

CELEB_REAL   = os.path.join(base, "Celeb-real")
YOUTUBE_REAL = os.path.join(base, "YouTube-real")
CELEB_FAKE   = os.path.join(base, "Celeb-synthesis")

# Merge Celeb-real + YouTube-real into one folder
MERGED_REAL = "/content/drive/MyDrive/celeb_df_raw/merged_real"
os.makedirs(MERGED_REAL, exist_ok=True)

print("Merging real video folders...")
for src_folder, tag in [(CELEB_REAL, "celeb"), (YOUTUBE_REAL, "yt")]:
    for f in os.listdir(src_folder):
        if f.endswith(".mp4"):
            dst = os.path.join(MERGED_REAL, f"{tag}_{f}")
            if not os.path.exists(dst):
                shutil.copy(os.path.join(src_folder, f), dst)

n_real = len([f for f in os.listdir(MERGED_REAL) if f.endswith(".mp4")])
n_fake = len([f for f in os.listdir(CELEB_FAKE)  if f.endswith(".mp4")])

print(f"Real videos : {n_real}")
print(f"Fake videos : {n_fake}")
print(f"Total       : {n_real + n_fake}")

RAW_REAL_DETECTED = MERGED_REAL
RAW_FAKE_DETECTED = CELEB_FAKE
print(f"\nRAW_REAL -> {RAW_REAL_DETECTED}")
print(f"RAW_FAKE -> {RAW_FAKE_DETECTED}")

## Step 4 — Configuration

In [ ]:
import torch
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 120

# Paths
RAW_REAL      = RAW_REAL_DETECTED
RAW_FAKE      = RAW_FAKE_DETECTED
PROCESSED_DIR = "/content/drive/MyDrive/celeb_df/processed"
SPLIT_DIR     = "/content/drive/MyDrive/celeb_df/splitted_data"
MODEL_PATH    = "/content/drive/MyDrive/celeb_df/best_model.pth"

# Hyperparameters
FRAME_LIMIT      = 100   # max frames extracted per video
FRAME_SKIP       = 2     # extract every Nth frame
FRAMES_PER_VIDEO = 30    # frames fed to model
BATCH_SIZE       = 12
EPOCHS           = 30
PATIENCE         = 5
LR               = 1e-4

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device      : {DEVICE}")
print(f"Batch size  : {BATCH_SIZE}")
print(f"Max epochs  : {EPOCHS}")
print(f"Patience    : {PATIENCE}")
print(f"LR          : {LR}")

for cat in ["real", "fake"]:
    os.makedirs(os.path.join(PROCESSED_DIR, cat), exist_ok=True)
    os.makedirs(os.path.join(SPLIT_DIR, cat), exist_ok=True)
print("Output directories ready.")

## Step 5 — Preprocessing: Face Extraction (MTCNN)
Each video is processed frame-by-frame. MTCNN detects and crops the face, then saves a **face-only video** (112×112px, 15fps).

> This step saves results to Drive — **safe to re-run**, already processed videos are skipped.

In [ ]:
import cv2
import numpy as np

# OpenCV built-in face detector - no extra packages needed
face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + "haarcascade_frontalface_default.xml")

def detect_face(frame_rgb):
    """Returns cropped face with padding, or None if no face found."""
    gray = cv2.cvtColor(frame_rgb, cv2.COLOR_RGB2GRAY)
    faces = face_cascade.detectMultiScale(gray, scaleFactor=1.1, minNeighbors=4, minSize=(60,60))
    if len(faces) == 0:
        return None
    x, y, w, h = sorted(faces, key=lambda f: f[2]*f[3], reverse=True)[0]
    pad = int(0.2 * max(w, h))
    x1 = max(0, x - pad)
    y1 = max(0, y - pad)
    x2 = min(frame_rgb.shape[1], x + w + pad)
    y2 = min(frame_rgb.shape[0], y + h + pad)
    return frame_rgb[y1:y2, x1:x2]

def process_video(src_path, dst_path, frame_limit=FRAME_LIMIT, frame_skip=FRAME_SKIP):
    cap = cv2.VideoCapture(src_path)
    face_frames, frame_idx, processed = [], 0, 0
    while cap.isOpened() and processed < frame_limit:
        cap.set(cv2.CAP_PROP_POS_FRAMES, frame_idx)
        ret, frame = cap.read()
        if not ret:
            break
        frame = cv2.resize(frame, (frame.shape[1]//2, frame.shape[0]//2))
        rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        face = detect_face(rgb)
        if face is not None and face.size > 0:
            face_bgr = cv2.cvtColor(cv2.resize(face, (112,112)), cv2.COLOR_RGB2BGR)
            face_frames.append(face_bgr)
            processed += 1
        frame_idx += frame_skip
    cap.release()
    if face_frames:
        writer = cv2.VideoWriter(dst_path, cv2.VideoWriter_fourcc(*"mp4v"), 15, (112,112))
        for f in face_frames:
            writer.write(f)
        writer.release()
        return True
    return False

def preprocess_folder(src_folder, dst_folder, label):
    videos = [f for f in os.listdir(src_folder) if f.endswith(".mp4")]
    ok, fail = 0, 0
    for i, v in enumerate(videos):
        dst = os.path.join(dst_folder, v)
        if os.path.exists(dst):
            ok += 1
            continue
        if process_video(os.path.join(src_folder, v), dst):
            ok += 1
        else:
            fail += 1
        if (i+1) % 100 == 0:
            print(f"  [{label}] {i+1}/{len(videos)} | saved={ok} no_face={fail}")
    print(f"[{label}] Done: {ok} saved, {fail} skipped (no face detected)")
    return ok

print("Processing REAL videos...")
n_real_proc = preprocess_folder(RAW_REAL, os.path.join(PROCESSED_DIR, "real"), "REAL")
print("Processing FAKE videos...")
n_fake_proc = preprocess_folder(RAW_FAKE, os.path.join(PROCESSED_DIR, "fake"), "FAKE")
print(f"Preprocessing complete! Real={n_real_proc}, Fake={n_fake_proc}")

### Preprocessing Result — Sample Face Crops

In [ ]:
import random
from PIL import Image

def get_sample_frames(video_path, n=5):
    cap = cv2.VideoCapture(video_path)
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    indices = np.linspace(0, total-1, n, dtype=int)
    frames = []
    for idx in indices:
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(idx))
        ret, frame = cap.read()
        if ret:
            frames.append(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
    cap.release()
    return frames

real_videos = [os.path.join(PROCESSED_DIR,"real",f) for f in os.listdir(os.path.join(PROCESSED_DIR,"real")) if f.endswith(".mp4")]
fake_videos = [os.path.join(PROCESSED_DIR,"fake",f) for f in os.listdir(os.path.join(PROCESSED_DIR,"fake")) if f.endswith(".mp4")]

n_show = 5
fig, axes = plt.subplots(2, n_show, figsize=(15, 6))
fig.suptitle("Sample Preprocessed Face Crops", fontsize=14, fontweight="bold")

for col, vpath in enumerate(random.sample(real_videos, n_show)):
    frames = get_sample_frames(vpath, 1)
    if frames:
        axes[0, col].imshow(frames[0])
    axes[0, col].set_title("REAL", color="green", fontsize=10)
    axes[0, col].axis("off")

for col, vpath in enumerate(random.sample(fake_videos, n_show)):
    frames = get_sample_frames(vpath, 1)
    if frames:
        axes[1, col].imshow(frames[0])
    axes[1, col].set_title("FAKE", color="red", fontsize=10)
    axes[1, col].axis("off")

plt.tight_layout()
plt.show()

## Step 6 — Data Splitting (70 / 20 / 10)
> Skip if already split.

In [ ]:
import shutil, random

def split_folder(src, dst_root, train_r=0.7, val_r=0.2):
    files = [f for f in os.listdir(src) if f.endswith('.mp4')]
    random.shuffle(files)
    n = len(files)
    splits = {
        'train': files[:int(n*train_r)],
        'val':   files[int(n*train_r):int(n*(train_r+val_r))],
        'test':  files[int(n*(train_r+val_r)):]
    }
    counts = {}
    for split, lst in splits.items():
        dest = os.path.join(dst_root, split)
        os.makedirs(dest, exist_ok=True)
        for f in lst:
            dst_file = os.path.join(dest, f)
            if not os.path.exists(dst_file):
                shutil.copy(os.path.join(src, f), dst_file)
        counts[split] = len(lst)
    return counts

print("Splitting REAL...")
real_counts = split_folder(os.path.join(PROCESSED_DIR,"real"), os.path.join(SPLIT_DIR,"real"))
print("Splitting FAKE...")
fake_counts = split_folder(os.path.join(PROCESSED_DIR,"fake"), os.path.join(SPLIT_DIR,"fake"))

print("\nSplit Summary:")
print(f"{'Split':<10} {'Real':>8} {'Fake':>8} {'Total':>8}")
print("-" * 36)
for split in ['train','val','test']:
    r, f = real_counts[split], fake_counts[split]
    print(f"{split:<10} {r:>8} {f:>8} {r+f:>8}")
print("Splitting complete!")

### Dataset Statistics Visualization

In [ ]:
splits = ['train', 'val', 'test']
real_ns = [real_counts[s] for s in splits]
fake_ns = [fake_counts[s] for s in splits]
total_ns = [r+f for r,f in zip(real_ns, fake_ns)]

x = range(len(splits))
width = 0.35

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle("Dataset Distribution", fontsize=14, fontweight="bold")

# Grouped bar chart
axes[0].bar([i - width/2 for i in x], real_ns, width, label='Real', color='#2ecc71')
axes[0].bar([i + width/2 for i in x], fake_ns, width, label='Fake', color='#e74c3c')
axes[0].set_xticks(list(x)); axes[0].set_xticklabels(splits)
axes[0].set_title("Real vs Fake per Split"); axes[0].set_ylabel("Videos")
axes[0].legend()
for i, (r, f) in enumerate(zip(real_ns, fake_ns)):
    axes[0].text(i-width/2, r+5, str(r), ha='center', fontsize=9)
    axes[0].text(i+width/2, f+5, str(f), ha='center', fontsize=9)

# Total per split
colors = ['#3498db','#9b59b6','#f39c12']
bars = axes[1].bar(splits, total_ns, color=colors)
axes[1].set_title("Total Videos per Split"); axes[1].set_ylabel("Videos")
for bar, val in zip(bars, total_ns):
    axes[1].text(bar.get_x()+bar.get_width()/2, bar.get_height()+5, str(val), ha='center', fontsize=10)

# Pie chart of overall real vs fake
total_real = sum(real_ns)
total_fake = sum(fake_ns)
axes[2].pie([total_real, total_fake], labels=['Real','Fake'],
            colors=['#2ecc71','#e74c3c'], autopct='%1.1f%%', startangle=90,
            textprops={'fontsize':12})
axes[2].set_title("Overall Class Balance")

plt.tight_layout()
plt.show()
print(f"Total dataset: {total_real + total_fake} videos ({total_real} real, {total_fake} fake)")

## Step 7 — Dataset & DataLoaders

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import timm
import seaborn as sns
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

train_transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.1),
    transforms.RandomGrayscale(p=0.05),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5,0.5,0.5], std=[0.5,0.5,0.5])
])
val_transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5,0.5,0.5], std=[0.5,0.5,0.5])
])

class DeepFakeDataset(Dataset):
    def __init__(self, folder, label, transform=None, fpv=FRAMES_PER_VIDEO):
        self.videos    = [os.path.join(folder,f) for f in os.listdir(folder) if f.endswith('.mp4')]
        self.label     = label
        self.transform = transform
        self.fpv       = fpv

    def __len__(self): return len(self.videos)

    def __getitem__(self, idx):
        cap = cv2.VideoCapture(self.videos[idx])
        frames = []
        while len(frames) < self.fpv and cap.isOpened():
            ret, frame = cap.read()
            if not ret: break
            frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            frame = cv2.resize(frame, (224, 224))
            if self.transform:
                frame = self.transform(frame)
            frames.append(frame)
        cap.release()
        while len(frames) < self.fpv:
            frames.append(torch.zeros(3, 224, 224))
        return torch.stack(frames), torch.tensor(self.label, dtype=torch.long)

def make_loader(split, tf, shuffle):
    real = DeepFakeDataset(os.path.join(SPLIT_DIR,"real",split), 0, tf)
    fake = DeepFakeDataset(os.path.join(SPLIT_DIR,"fake",split), 1, tf)
    ds = real + fake
    print(f"  {split:6}: {len(real)} real + {len(fake)} fake = {len(ds)} total")
    return DataLoader(ds, batch_size=BATCH_SIZE, shuffle=shuffle, num_workers=2, pin_memory=True)

print("Loading datasets...")
train_loader = make_loader("train", train_transform, True)
val_loader   = make_loader("val",   val_transform,   False)
test_loader  = make_loader("test",  val_transform,   False)
print("DataLoaders ready!")

### Augmentation Preview
Shows the same frame with different augmentations applied.

In [ ]:
real_vids = [os.path.join(SPLIT_DIR,"real","train",f)
             for f in os.listdir(os.path.join(SPLIT_DIR,"real","train")) if f.endswith(".mp4")]
sample_video = random.choice(real_vids)
cap = cv2.VideoCapture(sample_video)
ret, raw_frame = cap.read()
cap.release()
raw_rgb = cv2.cvtColor(raw_frame, cv2.COLOR_BGR2RGB)
raw_rgb = cv2.resize(raw_rgb, (224, 224))

from PIL import Image as PILImage
fig, axes = plt.subplots(2, 5, figsize=(16, 7))
fig.suptitle("Data Augmentation Examples (same frame, different augmentations)", fontsize=13, fontweight="bold")

axes[0,0].imshow(raw_rgb); axes[0,0].set_title("Original", fontsize=10); axes[0,0].axis("off")

for i in range(1, 10):
    aug = train_transform(raw_rgb)
    img = aug.permute(1,2,0).numpy()
    img = (img * 0.5 + 0.5).clip(0,1)
    r, c = divmod(i, 5)
    axes[r,c].imshow(img)
    axes[r,c].set_title(f"Aug #{i}", fontsize=10)
    axes[r,c].axis("off")

plt.tight_layout()
plt.show()

## Step 8 — Model Architecture
**EfficientNet-Lite0** extracts spatial features from each frame, then an **LSTM** models temporal patterns across the 30-frame sequence.

```
Input: (Batch, 30 frames, 3, 224, 224)
  └─> EfficientNet-Lite0 (per frame)  →  1280-dim feature
  └─> LSTM (2 layers, hidden=256)     →  temporal context
  └─> FC (256 → 128 → 2)             →  Real / Fake
```

In [ ]:
class DeepFakeDetector(nn.Module):
    def __init__(self):
        super().__init__()
        self.cnn = timm.create_model("efficientnet_lite0", pretrained=True)
        self.cnn.classifier = nn.Identity()
        self.dropout = nn.Dropout(0.4)
        self.lstm = nn.LSTM(input_size=1280, hidden_size=256, num_layers=2,
                            batch_first=True, dropout=0.3)
        self.fc = nn.Sequential(
            nn.Linear(256, 128), nn.ReLU(), nn.Dropout(0.3), nn.Linear(128, 2)
        )

    def forward(self, x):
        B, T, C, H, W = x.shape
        feats = self.cnn(x.view(B*T, C, H, W))
        feats = self.dropout(feats).view(B, T, -1)
        out, _ = self.lstm(feats)
        return self.fc(out[:, -1, :])

model = DeepFakeDetector().to(DEVICE)
total_params    = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total parameters     : {total_params:,}")
print(f"Trainable parameters : {trainable_params:,}")
print(f"Model device         : {DEVICE}")

### Model Architecture Diagram

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))
ax.axis("off")
fig.patch.set_facecolor("#1a1a2e")

blocks = [
    ("Input\n(B, 30, 3, 224, 224)", "#e74c3c"),
    ("EfficientNet-Lite0\n(per frame)\n1280-d features", "#e67e22"),
    ("Dropout\n(p=0.4)", "#f1c40f"),
    ("LSTM\n2 layers\nhidden=256", "#2ecc71"),
    ("FC Layer\n256→128→2\n+ Dropout", "#3498db"),
    ("Output\nReal / Fake", "#9b59b6"),
]

n = len(blocks)
for i, (label, color) in enumerate(blocks):
    x = i / (n-1)
    rect = plt.Rectangle((x - 0.07, 0.2), 0.14, 0.6, color=color, alpha=0.85, transform=ax.transAxes)
    ax.add_patch(rect)
    ax.text(x, 0.5, label, ha='center', va='center', fontsize=9, fontweight='bold',
            color='white', transform=ax.transAxes, wrap=True,
            bbox=dict(boxstyle='round,pad=0.1', facecolor='none', edgecolor='none'))
    if i < n-1:
        ax.annotate("", xy=((i+1)/(n-1) - 0.07, 0.5),
                    xytext=(x + 0.07, 0.5),
                    xycoords='axes fraction', textcoords='axes fraction',
                    arrowprops=dict(arrowstyle='->', color='white', lw=2))

ax.set_title("DeepFakeDetector Architecture", color='white', fontsize=14, fontweight='bold', pad=15)
plt.tight_layout()
plt.show()

## Step 9 — Training

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

use_amp = DEVICE == "cuda"
scaler  = torch.amp.GradScaler(enabled=use_amp)

train_losses, val_losses, train_accs, val_accs, lr_history = [], [], [], [], []
best_val_loss = float('inf')
patience_ctr  = 0

print(f"Training for up to {EPOCHS} epochs  |  Early stop patience={PATIENCE}")
print("=" * 80)

for epoch in range(EPOCHS):
    # ── Train ──────────────────────────────────────────────────────────────────
    model.train()
    t_loss, t_preds, t_labels = 0, [], []
    for videos, labels in train_loader:
        videos, labels = videos.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        with torch.amp.autocast(device_type=DEVICE, enabled=use_amp):
            out  = model(videos)
            loss = criterion(out, labels)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        t_loss += loss.item()
        t_preds.extend(torch.argmax(out,1).cpu().numpy())
        t_labels.extend(labels.cpu().numpy())

    train_losses.append(t_loss / len(train_loader))
    train_accs.append(accuracy_score(t_labels, t_preds))

    # ── Val ────────────────────────────────────────────────────────────────────
    model.eval()
    v_loss, v_preds, v_labels = 0, [], []
    with torch.no_grad():
        for videos, labels in val_loader:
            videos, labels = videos.to(DEVICE), labels.to(DEVICE)
            with torch.amp.autocast(device_type=DEVICE, enabled=use_amp):
                out  = model(videos)
                loss = criterion(out, labels)
            v_loss += loss.item()
            v_preds.extend(torch.argmax(out,1).cpu().numpy())
            v_labels.extend(labels.cpu().numpy())

    val_losses.append(v_loss / len(val_loader))
    val_accs.append(accuracy_score(v_labels, v_preds))
    lr_history.append(optimizer.param_groups[0]['lr'])
    scheduler.step()

    print(f"Epoch [{epoch+1:02}/{EPOCHS}]  "
          f"Train Loss: {train_losses[-1]:.4f}  Acc: {train_accs[-1]*100:.1f}%  |  "
          f"Val Loss: {val_losses[-1]:.4f}  Acc: {val_accs[-1]*100:.1f}%  "
          f"LR: {lr_history[-1]:.6f}")

    if v_loss < best_val_loss:
        best_val_loss = v_loss
        torch.save(model.state_dict(), MODEL_PATH)
        print("  -> Best model saved!")
        patience_ctr = 0
    else:
        patience_ctr += 1
        print(f"  -> No improvement ({patience_ctr}/{PATIENCE})")
        if patience_ctr >= PATIENCE:
            print(f"Early stopping triggered at epoch {epoch+1}")
            break

print("=" * 80)
print("Training complete!")

## Training Curves

In [ ]:
epochs_ran = range(1, len(train_losses)+1)
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle("Training Overview", fontsize=14, fontweight="bold")

# Loss
axes[0].plot(epochs_ran, train_losses, label='Train Loss', color='#e74c3c', linewidth=2)
axes[0].plot(epochs_ran, val_losses,   label='Val Loss',   color='#3498db', linewidth=2)
axes[0].fill_between(epochs_ran, train_losses, val_losses, alpha=0.1, color='gray')
axes[0].set_title("Loss"); axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("Loss")
axes[0].legend(); axes[0].grid(alpha=0.3)

# Accuracy
axes[1].plot(epochs_ran, [a*100 for a in train_accs], label='Train Acc', color='#e74c3c', linewidth=2)
axes[1].plot(epochs_ran, [a*100 for a in val_accs],   label='Val Acc',   color='#2ecc71', linewidth=2)
axes[1].axhline(max(val_accs)*100, linestyle='--', color='gray', alpha=0.5,
                label=f'Best Val: {max(val_accs)*100:.1f}%')
axes[1].set_title("Accuracy"); axes[1].set_xlabel("Epoch"); axes[1].set_ylabel("Accuracy (%)")
axes[1].legend(); axes[1].grid(alpha=0.3)

# Learning Rate
axes[2].plot(epochs_ran, lr_history, color='#9b59b6', linewidth=2)
axes[2].set_title("Learning Rate Schedule"); axes[2].set_xlabel("Epoch"); axes[2].set_ylabel("LR")
axes[2].grid(alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Best Val Accuracy : {max(val_accs)*100:.2f}%")
print(f"Best Val Loss     : {min(val_losses):.4f}")
print(f"Epochs trained    : {len(train_losses)}")

## Step 10 — Evaluation on Test Set

In [ ]:
from sklearn.metrics import (classification_report, confusion_matrix,
                              roc_curve, auc, precision_recall_curve)

model.load_state_dict(torch.load(MODEL_PATH, map_location=DEVICE))
model.eval()

all_preds, all_labels, all_probs = [], [], []
with torch.no_grad():
    for videos, labels in test_loader:
        videos, labels = videos.to(DEVICE), labels.to(DEVICE)
        out   = model(videos)
        probs = torch.softmax(out, dim=1)
        preds = torch.argmax(out, 1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
        all_probs.extend(probs[:,1].cpu().numpy())  # fake probability

print("Classification Report:")
print(classification_report(all_labels, all_preds, target_names=['Real','Fake'], digits=4))

### Evaluation Visualizations

In [ ]:
import numpy as np
from sklearn.metrics import roc_curve, auc, precision_recall_curve

fig, axes = plt.subplots(2, 2, figsize=(14, 11))
fig.suptitle("Model Evaluation — Test Set", fontsize=15, fontweight="bold")

# ── Confusion Matrix ───────────────────────────────────────────────────────────
cm = confusion_matrix(all_labels, all_preds)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0,0],
            xticklabels=['Real','Fake'], yticklabels=['Real','Fake'],
            annot_kws={"size": 14})
axes[0,0].set_xlabel('Predicted'); axes[0,0].set_ylabel('True')
axes[0,0].set_title('Confusion Matrix')
tn,fp,fn,tp = cm.ravel()
axes[0,0].set_xlabel(f'Predicted  |  Accuracy: {(tn+tp)/(tn+fp+fn+tp)*100:.2f}%')

# ── ROC Curve ─────────────────────────────────────────────────────────────────
fpr, tpr, _ = roc_curve(all_labels, all_probs)
roc_auc = auc(fpr, tpr)
axes[0,1].plot(fpr, tpr, color='#e74c3c', lw=2, label=f'ROC (AUC = {roc_auc:.4f})')
axes[0,1].plot([0,1],[0,1], 'k--', lw=1)
axes[0,1].fill_between(fpr, tpr, alpha=0.1, color='#e74c3c')
axes[0,1].set_xlabel('False Positive Rate'); axes[0,1].set_ylabel('True Positive Rate')
axes[0,1].set_title('ROC Curve'); axes[0,1].legend(); axes[0,1].grid(alpha=0.3)

# ── Precision-Recall Curve ────────────────────────────────────────────────────
prec, rec, _ = precision_recall_curve(all_labels, all_probs)
pr_auc = auc(rec, prec)
axes[1,0].plot(rec, prec, color='#2ecc71', lw=2, label=f'PR (AUC = {pr_auc:.4f})')
axes[1,0].fill_between(rec, prec, alpha=0.1, color='#2ecc71')
axes[1,0].set_xlabel('Recall'); axes[1,0].set_ylabel('Precision')
axes[1,0].set_title('Precision-Recall Curve'); axes[1,0].legend(); axes[1,0].grid(alpha=0.3)

# ── Confidence Distribution ───────────────────────────────────────────────────
probs_arr   = np.array(all_probs)
labels_arr  = np.array(all_labels)
axes[1,1].hist(probs_arr[labels_arr==0], bins=30, alpha=0.6, color='#2ecc71', label='Real videos')
axes[1,1].hist(probs_arr[labels_arr==1], bins=30, alpha=0.6, color='#e74c3c', label='Fake videos')
axes[1,1].axvline(0.5, color='black', linestyle='--', label='Threshold (0.5)')
axes[1,1].set_xlabel('Predicted Fake Probability')
axes[1,1].set_ylabel('Count')
axes[1,1].set_title('Confidence Distribution')
axes[1,1].legend(); axes[1,1].grid(alpha=0.3)

plt.tight_layout()
plt.show()
print(f"ROC-AUC  : {roc_auc:.4f}")
print(f"PR-AUC   : {pr_auc:.4f}")

### Per-Class Metrics

In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score

metrics = {
    'Precision': [precision_score(all_labels, all_preds, pos_label=0),
                  precision_score(all_labels, all_preds, pos_label=1)],
    'Recall':    [recall_score(all_labels, all_preds, pos_label=0),
                  recall_score(all_labels, all_preds, pos_label=1)],
    'F1-Score':  [f1_score(all_labels, all_preds, pos_label=0),
                  f1_score(all_labels, all_preds, pos_label=1)],
}

x = np.arange(2)
width = 0.25
colors = ['#3498db','#2ecc71','#e74c3c']
fig, ax = plt.subplots(figsize=(9, 5))
for i, (metric, vals) in enumerate(metrics.items()):
    bars = ax.bar(x + i*width, [v*100 for v in vals], width,
                  label=metric, color=colors[i], alpha=0.85)
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.5,
                f'{val*100:.1f}%', ha='center', va='bottom', fontsize=9)

ax.set_title("Per-Class Metrics", fontsize=13, fontweight='bold')
ax.set_xticks(x + width); ax.set_xticklabels(['Real','Fake'], fontsize=12)
ax.set_ylabel("Score (%)"); ax.set_ylim(0, 115)
ax.legend(); ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

### Sample Test Predictions — Correct vs Incorrect

In [ ]:
def get_video_prediction(video_path, label):
    cap = cv2.VideoCapture(video_path)
    frames, thumb = [], None
    while len(frames) < FRAMES_PER_VIDEO and cap.isOpened():
        ret, frame = cap.read()
        if not ret: break
        rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        if thumb is None:
            thumb = cv2.resize(rgb, (112,112))
        frames.append(val_transform(cv2.resize(rgb,(224,224))))
    cap.release()
    while len(frames) < FRAMES_PER_VIDEO:
        frames.append(torch.zeros(3,224,224))
    tensor = torch.stack(frames).unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        probs = torch.softmax(model(tensor), dim=1)[0]
    pred = torch.argmax(probs).item()
    return pred, probs[1].item(), thumb

# Collect sample predictions from test set
real_test = [os.path.join(SPLIT_DIR,"real","test",f)
             for f in os.listdir(os.path.join(SPLIT_DIR,"real","test")) if f.endswith(".mp4")]
fake_test = [os.path.join(SPLIT_DIR,"fake","test",f)
             for f in os.listdir(os.path.join(SPLIT_DIR,"fake","test")) if f.endswith(".mp4")]

samples = [(v,0) for v in random.sample(real_test, min(4,len(real_test)))] +           [(v,1) for v in random.sample(fake_test, min(4,len(fake_test)))]
random.shuffle(samples)

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
fig.suptitle("Sample Test Predictions", fontsize=14, fontweight="bold")
axes = axes.flatten()

for i, (vpath, true_label) in enumerate(samples[:8]):
    pred, fake_prob, thumb = get_video_prediction(vpath, true_label)
    correct = pred == true_label
    true_str = "REAL" if true_label == 0 else "FAKE"
    pred_str = "REAL" if pred == 0 else "FAKE"
    border_color = "#2ecc71" if correct else "#e74c3c"
    status = "CORRECT" if correct else "WRONG"

    if thumb is not None:
        axes[i].imshow(thumb)
    for spine in axes[i].spines.values():
        spine.set_edgecolor(border_color); spine.set_linewidth(4)
    axes[i].set_title(
        f"True: {true_str}\nPred: {pred_str} ({fake_prob*100:.1f}% fake)\n[{status}]",
        color=border_color, fontsize=9, fontweight='bold')
    axes[i].tick_params(left=False, bottom=False, labelleft=False, labelbottom=False)

plt.tight_layout()
plt.show()

## Step 11 — Predict on a New Video
Upload any `.mp4` video to get a **Real / Fake** verdict with confidence score and per-frame breakdown.

In [ ]:
from google.colab import files as colab_files
from torchvision import transforms as T

infer_tf = T.Compose([
    T.ToPILImage(),
    T.ToTensor(),
    T.Normalize(mean=[0.5,0.5,0.5], std=[0.5,0.5,0.5])
])

def predict_new_video(path):
    cap = cv2.VideoCapture(path)
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    indices = np.linspace(0, total-1, FRAMES_PER_VIDEO, dtype=int)
    frames, thumbnails = [], []
    for idx in indices:
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(idx))
        ret, frame = cap.read()
        if not ret: continue
        rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        thumbnails.append(cv2.resize(rgb,(112,112)))
        frames.append(infer_tf(cv2.resize(rgb,(224,224))))
    cap.release()
    while len(frames) < FRAMES_PER_VIDEO:
        frames.append(torch.zeros(3,224,224))

    tensor = torch.stack(frames).unsqueeze(0).to(DEVICE)
    model.eval()
    with torch.no_grad():
        probs = torch.softmax(model(tensor), dim=1)[0]

    pred  = torch.argmax(probs).item()
    label = "FAKE" if pred == 1 else "REAL"
    return label, probs[0].item(), probs[1].item(), thumbnails

model.load_state_dict(torch.load(MODEL_PATH, map_location=DEVICE))

print("Upload a video file (.mp4):")
uploaded   = colab_files.upload()
video_path = list(uploaded.keys())[0]
print(f"Uploaded: {video_path}")

label, real_p, fake_p, thumbs = predict_new_video(video_path)
conf = fake_p*100 if label == "FAKE" else real_p*100
color = "#e74c3c" if label == "FAKE" else "#2ecc71"

print()
print("=" * 44)
print(f"  VERDICT     : {label}")
print(f"  Confidence  : {conf:.2f}%")
print(f"  Real prob   : {real_p*100:.2f}%")
print(f"  Fake prob   : {fake_p*100:.2f}%")
print("=" * 44)

# Show thumbnail grid
n_show = min(10, len(thumbs))
fig, axes = plt.subplots(1, n_show, figsize=(18, 3))
fig.suptitle(f"Verdict: {label}  ({conf:.1f}% confident)",
             fontsize=13, fontweight="bold", color=color)
for i, ax in enumerate(axes):
    ax.imshow(thumbs[i * (len(thumbs)//n_show)])
    ax.axis("off")
plt.tight_layout()
plt.show()